## SQL With Spark

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [23]:
df_green = spark.read.parquet("Data/pq/green/*/*")
df_green.show(10)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|    NULL| 2020-01-03 10:23:00|  2020-01-03 10:48:00|              NULL|      NULL|          92|          75|           NULL|        10.37|      26.67|  0.0|    0.

In [24]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [25]:
df_yellow = spark.read.parquet("Data/pq/yellow/*/*")
df_yellow.show(6)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       2| 2020-01-16 13:25:44|  2020-01-16 13:38:16|              1|         1.88|         1|                 N|         239|         263|           1|       10.0|  0.0|    0.5|      3.33|         0.0|                  0.3

In [26]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [27]:
df_green.columns

['VendorID',
 'lpep_pickup_datetime',
 'lpep_dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge']

In [28]:
df_green = df_green.withColumnRenamed(
    "lpep_pickup_datetime", "pickup_datetime"
).withColumnRenamed(
    "lpep_dropoff_datetime", "dropoff_datetime"
)

In [29]:
df_yellow = df_yellow.withColumnRenamed(
    "tpep_pickup_datetime", "pickup_datetime"
).withColumnRenamed(
    "tpep_dropoff_datetime", "dropoff_datetime"
)

In [30]:
set(df_green.columns) & set(df_yellow.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'dropoff_datetime',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'pickup_datetime',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance'}

In [31]:
common_col = []
for col in df_green.columns:
    if col in df_yellow.columns:
        common_col.append(col)
    else:
        continue

In [32]:
common_col

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge']

In [36]:
from pyspark.sql.functions import lit, expr

In [38]:
df_green.select(
    expr("*"),
    lit("green").alias("service_type")
).show(10)

+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------+
|VendorID|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|service_type|
+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------+
|    NULL|2020-01-03 10:23:00|2020-01-03 10:48:00|              NULL|      NULL|          92|          75|           NULL|        10.

In [42]:
from pyspark.sql.functions import lit

df_green.select(
    "*",
    lit("green").alias("service_type")
).show(10)

+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------+
|VendorID|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|service_type|
+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------+
|    NULL|2020-01-03 10:23:00|2020-01-03 10:48:00|              NULL|      NULL|          92|          75|           NULL|        10.

In [41]:
df_green.selectExpr(
    "*",
    "'green' as service_type"
).show(10)

+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------+
|VendorID|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|service_type|
+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------+
|    NULL|2020-01-03 10:23:00|2020-01-03 10:48:00|              NULL|      NULL|          92|          75|           NULL|        10.

In [43]:
df_green = df_green.select(common_col)\
           .withColumn("service_type", lit("green"))

df_yellow = df_yellow.select(common_col)\
            .withColumn("service_type", lit("yellow"))

df_trips_data = df_green.unionAll(df_yellow)

df_trips_data.groupBy("service_type").count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



```python
df_trips_data.registerTempTable("trips_data")
```

In [44]:
df_trips_data.registerTempTable("trips_data")

C:\Spark\spark-4.1.1\spark-4.1.1-bin-hadoop3\python\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [45]:
df_trips_data.createOrReplaceTempView("trips_data")

In [47]:
def read_sql(exp: str):
    return spark.sql(exp).show()

read_sql("""
  SELECT * FROM trips_data
   LIMIT 10
""")

+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+--------------------+------------+
|VendorID|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|payment_type|congestion_surcharge|service_type|
+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+--------------------+------------+
|    NULL|2020-01-03 10:23:00|2020-01-03 10:48:00|              NULL|      NULL|          92|          75|           NULL|        10.37|      26.67|  0.0|    0.0|       0.0|        6.12|       

In [49]:
read_sql(
    """
    SELECT 
        service_type, 
        count(*)
    from trips_data
    group by 
        service_type
    """
)

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [50]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [51]:
df_result.show()

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|         127

In [52]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [3]:
df_green = spark.read.parquet('data/pq/green/*/*')
df_green.show(6)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|    NULL| 2020-01-03 10:23:00|  2020-01-03 10:48:00|              NULL|      NULL|          92|          75|           NULL|        10.37|      26.67|  0.0|    0.

In [4]:
import warnings
warnings.filterwarnings("ignore")
df_green.registerTempTable('green')

In [5]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [6]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

In [7]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')
df_yellow.registerTempTable('yellow')

In [8]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [9]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

In [10]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [11]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [12]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [13]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [14]:
df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer').explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [coalesce(hour#130, hour#134) AS hour#146, coalesce(zone#131, zone#135) AS zone#147, green_amount#139, green_number_records#140L, yellow_amount#142, yellow_number_records#143L]
   +- SortMergeJoin [hour#130, zone#131], [hour#134, zone#135], FullOuter
      :- Sort [hour#130 ASC NULLS FIRST, zone#131 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(hour#130, zone#131, 200), ENSURE_REQUIREMENTS, [plan_id=415]
      :     +- Project [hour#130, zone#131, amount#132 AS green_amount#139, number_records#133L AS green_number_records#140L]
      :        +- FileScan parquet [hour#130,zone#131,amount#132,number_records#133L] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/user/Documents/Data-Engineering-Zoomcamp/spark/data/rep..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<hour:timestamp,zone:int,amount:double,number_records:bigint>
      +- Sor

In [15]:
df_join = spark.read.parquet('data/report/revenue/total')

df_zones = spark.read.csv('taxi_zone_lookup.csv', header="true", inferSchema="true")

In [16]:
df_join.show(6)

+-------------------+----+------------------+--------------------+-----------------+---------------------+
|               hour|zone|      green_amount|green_number_records|    yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+-----------------+---------------------+
|2020-01-01 00:00:00|  13|              NULL|                NULL|           1214.8|                   56|
|2020-01-01 00:00:00|  15|              NULL|                NULL|            34.09|                    1|
|2020-01-01 00:00:00|  45|              NULL|                NULL|732.4800000000002|                   42|
|2020-01-01 00:00:00|  47|              13.3|                   1|              8.3|                    1|
|2020-01-01 00:00:00|  62|             15.95|                   1|            61.43|                    1|
|2020-01-01 00:00:00|  70|54.900000000000006|                   3|              9.3|                    1|
+-------------------+----+-----------

In [17]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [21]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones', mode='overwrite')

In [22]:
df_result.count()

1803705

In [23]:
df_join.count()

1803705

In [24]:
df_join.join(df_zones, df_join.zone == df_zones.LocationID).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [zone#149], [LocationID#171], Inner, BuildRight, false
   :- Filter isnotnull(zone#149)
   :  +- FileScan parquet [hour#148,zone#149,green_amount#150,green_number_records#151L,yellow_amount#152,yellow_number_records#153L] Batched: true, DataFilters: [isnotnull(zone#149)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/user/Documents/Data-Engineering-Zoomcamp/spark/data/rep..., PartitionFilters: [], PushedFilters: [IsNotNull(zone)], ReadSchema: struct<hour:timestamp,zone:int,green_amount:double,green_number_records:bigint,yellow_amount:doub...
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=950]
      +- Filter isnotnull(LocationID#171)
         +- FileScan csv [LocationID#171,Borough#172,Zone#173,service_zone#174] Batched: false, DataFilters: [isnotnull(LocationID#171)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:

# Spark Join Strategies: Sort Merge Join (SMJ) vs Broadcast Hash Join (BHJ)

## Overview

In **Apache Spark**, joins are one of the most expensive operations because they involve:

- data movement
- computation
- memory usage
- network communication across a distributed cluster

Spark uses different **join strategies** depending on:

- dataset size
- cluster memory
- join keys
- Spark configuration
- table statistics
- Catalyst Optimizer
- Adaptive Query Execution (AQE)

The two most common join strategies are:

- **Sort Merge Join (SMJ)** → Large vs Large datasets
- **Broadcast Hash Join (BHJ)** → Small vs Large datasets

Understanding these join strategies is critical for **Data Engineers** because they directly impact:

- performance
- cost
- execution time
- cluster resource usage
- scalability

---

# 1. Sort Merge Join (SMJ)

## Definition

**Sort Merge Join** is a Spark join strategy where:

> Spark shuffles both datasets on the join key, sorts them, and merges them together.

It is mainly used when **both datasets are large** and cannot be broadcast.

---

## How Sort Merge Join Works

### Step 1: Read Data

Spark reads both datasets from storage.

```
FileScan
```

---

### Step 2: Shuffle Data

Spark redistributes data across executors.

```
Exchange hashpartitioning(join_key)
```

This ensures rows with the same join key go to the same executor.

---

### Step 3: Sort Partitions

Each partition is sorted by the join key.

```
Sort [join_key ASC]
```

---

### Step 4: Merge Join

Spark scans both sorted datasets and merges them.

```
SortMergeJoin
```

---

## Execution Flow

```
Table A → Shuffle → Sort
                        → Merge Join → Output
Table B → Shuffle → Sort
```

---

## Characteristics

### Advantages

- Highly scalable
- Memory efficient
- Suitable for distributed systems
- Works well with large datasets
- Stable for big data pipelines

### Disadvantages

- Shuffle is expensive
- Sorting is CPU intensive
- High network cost
- Slower than broadcast join

---

## When Spark Uses Sort Merge Join

Spark chooses SMJ when:

- both tables are large
- broadcast is not possible
- join keys exist
- distributed execution is required
- broadcast threshold is exceeded

---

## Example Physical Plan

```
Exchange
Sort
SortMergeJoin
```

---

## Real-World Use Case

Large data lake join:

```
transactions (300GB)
clickstream (400GB)
```

Best strategy:

```
Sort Merge Join
```

Used in:

- ETL pipelines
- data lakes
- fact-to-fact joins
- large aggregations
- warehouse transformations

---

# 2. Broadcast Hash Join (BHJ)

## Definition

**Broadcast Hash Join** is a Spark join strategy where:

> Spark broadcasts the small dataset to all executors and performs a local hash join.

It is used when **one dataset is small** and fits in memory.

---

## How Broadcast Join Works

### Step 1: Identify Small Table

Spark checks:

```
spark.sql.autoBroadcastJoinThreshold
```

Default:

```
10MB
```

---

### Step 2: Broadcast Small Table

Spark sends the small table to all executors.

```
BroadcastExchange
```

---

### Step 3: Build Hash Table

Spark creates a hash map:

```
join_key → row
```

---

### Step 4: Scan Large Table

Executors scan the large dataset and perform local join.

```
BroadcastHashJoin
```

---

## Execution Flow

```
Small Table → Broadcast → All Executors

Large Table → Scan → Local Join → Output
```

---

## Characteristics

### Advantages

- Very fast
- No shuffle
- No sorting
- Low network cost
- Efficient for fact-dimension joins

### Disadvantages

- Requires memory
- Small table must fit in executor memory
- Not suitable for large datasets
- Risk of Out Of Memory (OOM)

---

## When Spark Uses Broadcast Join

Spark chooses BHJ when:

- one table is small
- table size < broadcast threshold
- enough executor memory exists
- join is equi-join
- broadcast is enabled

---

## Example Physical Plan

```
BroadcastExchange
BroadcastHashJoin
```

---

## Real-World Use Case

Data warehouse scenario:

```
transactions (500GB)
customers (5MB)
```

Best strategy:

```
Broadcast Join
```

Used in:

- fact-dimension joins
- lookup tables
- reference tables
- star schema
- reporting pipelines

---

# Key Differences

| Feature | Sort Merge Join | Broadcast Hash Join |
|--------|----------------|--------------------|
| Dataset Size | Large vs Large | Small vs Large |
| Shuffle | Yes | No |
| Sorting | Yes | No |
| Speed | Medium | Very Fast |
| Memory Usage | Low | High |
| Network Cost | High | Low |
| Scalability | Very High | Limited by Memory |
| CPU Cost | High | Low |
| Best Use Case | Big Data Joins | Dimension Joins |

---

# Performance Comparison

## Sort Merge Join

```
Shuffle → Sort → Merge
```

Time Complexity:

```
O(n log n)
```

---

## Broadcast Hash Join

```
Broadcast → Hash Lookup
```

Time Complexity:

```
O(n)
```

---

# Configuration

## Check Broadcast Threshold

```python
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
```

---

## Change Broadcast Threshold

```python
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50MB")
```

---

## Disable Broadcast Join

```python
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
```

---

# Force Broadcast Join

```python
from pyspark.sql.functions import broadcast

df_large.join(broadcast(df_small), "id")
```

---

# Force Sort Merge Join

```python
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
```

---

# Adaptive Query Execution (AQE)

Spark can change join strategy at runtime.

```
AdaptiveSparkPlan
```

Example:

```
Broadcast Join → Sort Merge Join
```

If small table becomes large.

### AQE improves

- performance
- memory usage
- runtime optimization
- execution planning

---

# Best Practices

## Use Broadcast Join When

- dimension tables
- lookup tables
- small datasets
- star schema
- warehouse joins

---

## Use Sort Merge Join When

- large datasets
- fact-to-fact joins
- big data pipelines
- data lakes
- heavy ETL

---

# Data Engineering Rule

### Broadcast Join

```
small table + large table
```

### Sort Merge Join

```
large table + large table
```

---

# Conclusion

## Broadcast Hash Join

- fast
- efficient
- avoids shuffle
- best for small tables

## Sort Merge Join

- scalable
- stable
- distributed-friendly
- best for large datasets

Understanding both strategies helps Data Engineers:

- optimize Spark jobs
- reduce cost
- improve performance
- design efficient pipelines
- control cluster resource usage

---

# Author

**Olawoye Taofeek**  
Data Engineer | Spark | AWS | Data Warehouse | Big Data

## Working with DATE and TIMESTAMP IN Pyspark

In [25]:
from pyspark.sql.functions import current_date, current_timestamp
dateDF = spark.range(10)\
        .withColumn("today", current_date())\
        .withColumn("now", current_timestamp())
dateDF.createOrReplaceTempView("dateTable")
dateDF.printSchema()

root
 |-- id: long (nullable = false)
 |-- today: date (nullable = false)
 |-- now: timestamp (nullable = false)



To set Time zone:
> `spark.conf.sessionLocalTimeZone`

In [26]:
spark.sql("""SELECT * FROM dateTable""").show()

+---+----------+--------------------+
| id|     today|                 now|
+---+----------+--------------------+
|  0|2026-03-29|2026-03-29 07:36:...|
|  1|2026-03-29|2026-03-29 07:36:...|
|  2|2026-03-29|2026-03-29 07:36:...|
|  3|2026-03-29|2026-03-29 07:36:...|
|  4|2026-03-29|2026-03-29 07:36:...|
|  5|2026-03-29|2026-03-29 07:36:...|
|  6|2026-03-29|2026-03-29 07:36:...|
|  7|2026-03-29|2026-03-29 07:36:...|
|  8|2026-03-29|2026-03-29 07:36:...|
|  9|2026-03-29|2026-03-29 07:36:...|
+---+----------+--------------------+



In [29]:
## Adding and subtracting dates

from pyspark.sql.functions import date_add, date_sub, col
dateDF.select(date_sub(col("today"), 5).alias("5_days_ago"), date_add("today", 5).alias("in_5_days")).show()

+----------+----------+
|5_days_ago| in_5_days|
+----------+----------+
|2026-03-24|2026-04-03|
|2026-03-24|2026-04-03|
|2026-03-24|2026-04-03|
|2026-03-24|2026-04-03|
|2026-03-24|2026-04-03|
|2026-03-24|2026-04-03|
|2026-03-24|2026-04-03|
|2026-03-24|2026-04-03|
|2026-03-24|2026-04-03|
|2026-03-24|2026-04-03|
+----------+----------+



Another common task is to take a look at the difference between two dates. We can do this with the `datediff` function that will return the number of days in between two dates. Most often we just care about the days, and because the number of days varies from month to month, there also exists a function, `months_between`, that gives you the number of months between two dates:

In [30]:
# in Python
from pyspark.sql.functions import datediff, months_between, to_date
dateDF.withColumn("week_ago", date_sub(col("today"), 7))\
    .select(datediff(col("week_ago"), col("today"))).show(1)

+-------------------------+
|datediff(week_ago, today)|
+-------------------------+
|                       -7|
+-------------------------+
only showing top 1 row


In [32]:
from pyspark.sql.functions import lit

In [33]:
dateDF.select(
    to_date(lit("2016-01-01")).alias("start"),
    to_date(lit("2017-05-22")).alias("end"))\
    .select(months_between(col("start"), col("end"))).show(1)

+--------------------------------+
|months_between(start, end, true)|
+--------------------------------+
|                    -16.67741935|
+--------------------------------+
only showing top 1 row


In [34]:
spark.sql("""
        SELECT to_date('2016-01-01'), 
               months_between('2016-01-01', '2017-01-01'),
               datediff('2016-01-01', '2017-01-01')
        FROM dateTable"""
    )\
    .show()

+-------------------+--------------------------------------------+--------------------------------+
|to_date(2016-01-01)|months_between(2016-01-01, 2017-01-01, true)|datediff(2016-01-01, 2017-01-01)|
+-------------------+--------------------------------------------+--------------------------------+
|         2016-01-01|                                       -12.0|                            -366|
|         2016-01-01|                                       -12.0|                            -366|
|         2016-01-01|                                       -12.0|                            -366|
|         2016-01-01|                                       -12.0|                            -366|
|         2016-01-01|                                       -12.0|                            -366|
|         2016-01-01|                                       -12.0|                            -366|
|         2016-01-01|                                       -12.0|                            -366|


In [35]:
## Spark will not throw an error if it cannot parse the date; rather, it will just return null.

dateDF.select(to_date(lit("2016-20-12")),to_date(lit("2017-12-11"))).show(1)

{"ts": "2026-03-29 07:57:56.052", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '2016-20-12' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "to_date", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o221.showString.\n: org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value '2016-20-12' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"to_date\" was called from\nj

DateTimeException: [CAST_INVALID_INPUT] The value '2016-20-12' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"to_date" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)


In [36]:
from pyspark.sql.functions import to_date
dateFormat = "yyyy-dd-MM"
cleanDateDF = spark.range(1).select(
    to_date(lit("2017-12-11"), dateFormat).alias("date"),
    to_date(lit("2017-20-12"), dateFormat).alias("date2")
)
cleanDateDF.show(1)

+----------+----------+
|      date|     date2|
+----------+----------+
|2017-11-12|2017-12-20|
+----------+----------+



### Working with NULLS

In [ ]:
# coalesce: returns the first non-null value among its arguments.
from pyspark.sql.functions import coalesce
df.select(coalesce(col("Description"), col("CustomerId"))).show()

#### ifnull, nullIf, nvl, and nvl2
There are several other SQL functions that you can use to achieve similar things. ifnull allows you to select the second value if the first is null, and defaults to the first. Alternatively, you could use nullif, which returns null if the two values are equal or else returns the second if they are not. nvl returns the second value if the first is null, but defaults to the first. Finally, nvl2 returns the second value if the first is not null; otherwise, it will return the last specified value (else_value in the following example):


In [39]:
spark.sql("""
  SELECT
    ifnull(null, 'return_value'),
    nullif('value', 'value'),
    nvl(null, 'return_value'),
    nvl2('not_null', 'return_value', "else_value")
 FROM dateTable LIMIT 1
""").show()

+--------------------------+--------------------+-----------------------+----------------------------------------+
|ifnull(NULL, return_value)|nullif(value, value)|nvl(NULL, return_value)|nvl2(not_null, return_value, else_value)|
+--------------------------+--------------------+-----------------------+----------------------------------------+
|              return_value|                NULL|           return_value|                            return_value|
+--------------------------+--------------------+-----------------------+----------------------------------------+

